# CNN_VIT_BILSTM_CROSS_ATTENTION_BASED_TRAFFIC_MANAGEMENT_SYSTEM
## S17 - TrafficCAM arm, and the first honest cross-camera test (P21)

**Why this run exists.** `s15_yolov8s_joint_aug` scores **mAP50 0.8941** on
BMD-45 elevated, which has been this project's detection headline. Evaluated
unchanged on TrafficCAM - human-annotated Indian elevated CCTV from *other*
cities - it scores **0.3500**.

Two causes, both measured before this notebook was written:

**`e_rickshaw` was never trained.** IDD contains **zero** e_rickshaw instances
in train, val and test. The detector carries an output head that has never seen
one example. Matched by IoU against 168 ground-truth e-rickshaws, it calls
47.6% of them `motorcycle`, 25.6% `auto_rickshaw`, and misses 23.8% - so 76% are
**localised correctly and labelled wrong**. The features exist; the class needs
examples.

**The headline is measured on easy objects.** Median linear box size: BMD-45 car
**0.1055**, IDD 0.0564, TrafficCAM 0.0556. BMD-45's objects are twice the linear
size - four times the area - of both other sets, which agree with each other.

**So this run answers a question the S16 distillation arm could not:** does
adding human-annotated data from other Indian cameras fix `e_rickshaw` and close
the cross-camera gap, and what does it cost on the sets we already report?

Every result below is reported on **three** test sets, never one.

In [ ]:
import os, sys, random
SEED = 42
random.seed(SEED); os.environ["PYTHONHASHSEED"] = str(SEED)

import numpy as np, torch
np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit("NO GPU. Settings -> Accelerator -> GPU T4 x2.")
print("gpu  ", torch.cuda.get_device_name(0))

# `kernels push` resets the accelerator to P100, which is sm_60; current PyTorch
# builds start at sm_70, so the card cannot run at all. Fail in fifteen seconds
# rather than an hour into the weekly quota.
major, minor = torch.cuda.get_device_capability(0)
supported = torch.cuda.get_arch_list()
print("arch  ", "sm_%d%d" % (major, minor), "| build supports", supported)
if ("sm_%d%d" % (major, minor)) not in supported:
    raise SystemExit(
        "INCOMPATIBLE GPU sm_%d%d; build supports %s. "
        "Settings -> Accelerator -> GPU T4 x2 (sm_75)." % (major, minor, supported)
    )

## The code comes from the repository, not from this notebook

In [ ]:
!git clone --depth 1 https://github.com/Divyansh-9/CNN_VIT_BILSTM_CROSS_ATTENTION_BASED_TRAFFIC_MANAGEMENT_SYSTEM.git /kaggle/working/repo 2>&1 | tail -2
%cd /kaggle/working/repo
!pip install -q ultralytics huggingface_hub

## BMD-45 - kept in the training union

Dropping it would confound the experiment: any change on elevated data could be
the missing source rather than the added one.

In [ ]:
COUNT = 8000     # matched to the IDD subsample, as S14
import subprocess
subprocess.run([
    "python", "scripts/prepare_bmd45.py",
    "--count", str(COUNT), "--workers", "16",
    "--max-failure-rate", "0.02",
    "--out", "/kaggle/working/bmd45_yolo",
], check=True)

## Locate the attached datasets

In [ ]:
from pathlib import Path
INPUT = Path("/kaggle/input")
candidates = sorted(INPUT.iterdir()) if INPUT.exists() else []
print("mounted:", [c.name for c in candidates] or "NOTHING")

def find_yolo_root(base):
    for path in [base, *base.rglob("*")]:
        if path.is_dir() and (path / "images" / "train").is_dir():
            return path
    return None

roots = {}
for c in candidates:
    r = find_yolo_root(c)
    if r is None:
        continue
    # TrafficCAM ships its own data.yaml; IDD does too. Tell them apart by name.
    key = "trafficcam" if "trafficcam" in c.name.lower() else "idd"
    roots.setdefault(key, r)

for key in ("idd", "trafficcam"):
    if key not in roots:
        raise SystemExit(
            "%s not attached. Attach BOTH datasets in the Input panel - a UI "
            "Save & Run All uses the draft attachments, not kernel-metadata." % key
        )
IDD, TCAM = roots["idd"], roots["trafficcam"]
print("IDD       ", IDD)
print("TrafficCAM", TCAM)

CKPT = None
for c in candidates:
    for found in c.rglob("s15_*_best.pt"):
        CKPT = found
        break
    if CKPT is not None:
        break
if CKPT is None:
    raise SystemExit("S15 checkpoint not found (ships with the pseudo-label dataset)")
print("CKPT      ", CKPT)

## Compose - three evaluation sets, one training union

`eval_bmd45.yaml` and `eval_idd.yaml` are written by the same script S14 and S15
used and are **not** regenerated with TrafficCAM in them. `eval_trafficcam.yaml`
points at TrafficCAM's own test split, grouped by camera session.

In [ ]:
import subprocess, yaml
from pathlib import Path

# BMD-45 + IDD, exactly as S14/S15 composed them. This also writes the two
# evaluation configs those runs were scored on, and they are NOT regenerated
# with TrafficCAM in them.
subprocess.run([
    "python", "scripts/build_joint_dataset.py",
    "--bmd45", "/kaggle/working/bmd45_yolo",
    "--idd", str(IDD),
    "--out", "/kaggle/working/joint",
], check=True)

CLASSES = ["car", "motorcycle", "auto_rickshaw", "e_rickshaw",
           "bus", "truck", "pedestrian", "cattle"]

# Training union: the S15 sources plus TrafficCAM train. Expressed as a config,
# not by copying gigabytes.
train_dirs = [str(Path("/kaggle/working/bmd45_yolo") / "images" / "train"),
              str(Path(IDD) / "images" / "train"),
              str(Path(TCAM) / "images" / "train")]
Path("/kaggle/working/s17").mkdir(parents=True, exist_ok=True)
Path("/kaggle/working/s17/train.yaml").write_text(
    "train:\n" + "".join("  - %s\n" % d for d in train_dirs)
    + "val: %s\n" % (Path(IDD) / "images" / "val")
    + "nc: %d\nnames: %s\n" % (len(CLASSES), CLASSES))

# Third evaluation config: TrafficCAM's own test split, grouped by camera
# session so no session appears in both train and test.
Path("/kaggle/working/s17/eval_trafficcam.yaml").write_text(
    "path: %s\ntrain: images/train\nval: images/val\ntest: images/test\n"
    "nc: %d\nnames: %s\n" % (TCAM, len(CLASSES), CLASSES))

print(Path("/kaggle/working/s17/train.yaml").read_text())

## Train - from the S15 checkpoint, one variable changed

In [ ]:
from ultralytics import YOLO

# Started from the S15 checkpoint and everything except the data held at S15's
# settings - same architecture, imgsz, batch, seed, and the A31 geometric
# augmentation. TrafficCAM in the training union is the only variable.
model = YOLO(str(CKPT))
results = model.train(
    data="/kaggle/working/s17/train.yaml",
    epochs=40, imgsz=640, batch=16, seed=SEED, patience=15,
    perspective=0.0006, degrees=8.0, shear=4.0,
    project="/kaggle/working/runs", name="s17_trafficcam", exist_ok=True, plots=True,
)

## Evaluate - on all three, separately

In [ ]:
WEIGHTS = "/kaggle/working/runs/s17_trafficcam/weights/best.pt"
import subprocess

BAR = "=" * 66
targets = [
    ("bmd45", "elevated CCTV, LARGE objects", "/kaggle/working/joint/eval_bmd45.yaml"),
    ("idd", "dashcam", "/kaggle/working/joint/eval_idd.yaml"),
    ("trafficcam", "elevated CCTV, other cities", "/kaggle/working/s17/eval_trafficcam.yaml"),
]
for tag, what, data in targets:
    print(); print(BAR); print("  %s  (%s)" % (tag.upper(), what)); print(BAR)
    subprocess.run([
        "python", "scripts/verify_detector_metrics.py",
        "--weights", WEIGHTS, "--data", data,
        "--split", "test", "--device", "0",
        "--out", "/kaggle/working/s17_metrics_%s.csv" % tag,
    ], check=True)

## S17 acceptance gate

Criterion 2 is the one this run exists for. `e_rickshaw` has never scored above
zero because it was never trained; TrafficCAM supplies 687 train boxes and 227
test boxes, so a real number is possible for the first time.

Criteria 3, 4 and 5 are what it must not break - especially `cattle`, which is
**absent from TrafficCAM entirely** and can only be forgotten, never improved,
by this run.

In [ ]:
import csv, statistics, time
import numpy as np
from ultralytics import YOLO

# Baselines: what s15 scores on each set today. The TrafficCAM figure is the one
# this run exists to move; the other two are what it must not break.
BASE = {"bmd45": 0.8941, "idd": 0.7104, "trafficcam": 0.3500}
BASE_ERICKSHAW = 0.000       # never trained — IDD has zero instances
BASE_CATTLE = 0.3516         # S14, IDD test, 183 boxes

def read(path):
    rows = [r for r in csv.DictReader(open(path)) if r.get("evaluated") == "True"]
    return rows, statistics.fmean(float(r["mAP50"]) for r in rows if r["mAP50"])

got, rows = {}, {}
for tag, _, _ in targets:
    rows[tag], got[tag] = read("/kaggle/working/s17_metrics_%s.csv" % tag)

def ap(tag, name):
    for r in rows[tag]:
        if r["class"] == name:
            return float(r["mAP50"])
    return None

erick = ap("trafficcam", "e_rickshaw")
cattle = ap("idd", "cattle")

student = YOLO(WEIGHTS)
frame = np.random.randint(0, 255, (1080, 1920, 3), dtype=np.uint8)
student.predict(source=frame, imgsz=640, verbose=False)
t0 = time.perf_counter()
for _ in range(20):
    student.predict(source=frame, imgsz=640, verbose=False)
fps = 20 / (time.perf_counter() - t0)

BAR = "=" * 66
print(BAR); print("  S17 ACCEPTANCE GATE  (fixed before this run)"); print(BAR)
checks = [
    ("1  TrafficCAM mAP50 > 0.3500 (the point of this run)",
     "%.4f (was %.4f, %+.4f)" % (got["trafficcam"], BASE["trafficcam"],
                                 got["trafficcam"] - BASE["trafficcam"]),
     got["trafficcam"] > BASE["trafficcam"]),
    ("2  e_rickshaw AP50 > 0.30 — the class that was never trained",
     ("%.4f (was 0.000, never trained)" % erick) if erick is not None
     else "NOT EVALUATED",
     erick is not None and erick > 0.30),
    ("3  BMD-45 mAP50 not down > 0.02",
     "%.4f (was %.4f, %+.4f)" % (got["bmd45"], BASE["bmd45"],
                                 got["bmd45"] - BASE["bmd45"]),
     got["bmd45"] >= BASE["bmd45"] - 0.02),
    ("4  IDD mAP50 not down > 0.02",
     "%.4f (was %.4f, %+.4f)" % (got["idd"], BASE["idd"],
                                 got["idd"] - BASE["idd"]),
     got["idd"] >= BASE["idd"] - 0.02),
    ("5  cattle AP50 not down > 0.02 (absent from TrafficCAM)",
     ("%.4f (was %.4f, %+.4f)" % (cattle, BASE_CATTLE, cattle - BASE_CATTLE))
     if cattle is not None else "NOT EVALUATED",
     cattle is not None and cattle >= BASE_CATTLE - 0.02),
    ("6  >= 10 fps on this host (ADR-003)",
     "%.1f fps on %s" % (fps, torch.cuda.get_device_name(0)),
     fps >= 10.0),
]
for label, value, ok in checks:
    print("  [%s]  %s" % ("PASS" if ok else "FAIL", label))
    print("          %s" % value)
print(BAR)
if all(ok for _, _, ok in checks):
    print("  ALL CRITERIA MET.")
else:
    print("  NOT ADOPTED. Record the numbers and the reason; do not retune the")
    print("  criteria to fit the result (ADR-012 discipline).")
print(BAR)
print()
print("  Report all three sets together, always, with object scale stated.")
print("  P21: a single mAP number was shown to be misleading — BMD-45's objects")
print("  are twice the linear size of IDD's and TrafficCAM's.")